In [14]:
import pandas as pd
from datasets import load_dataset

In [15]:
import pandas as pd

# 1. This link allows Colab to download the file directly from Drive
file_id = '1AIjpubvoxiJOMz974J6omBMg82UDN0OW'
url = f'https://drive.google.com/uc?id={file_id}'

# 2. Read the CSV
df = pd.read_csv(url)

# 3. View the first few rows to make sure it worked
df.head()


,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN
3,3,I remember going to the fireworks with my best...,sentimental,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.,NaN,NaN
4,4,I remember going to the fireworks with my best...,sentimental,Customer :Where has she gone?\nAgent :,We no longer talk.,NaN,NaN


In [16]:
df.head()

,Unnamed: 0,Situation,emotion,empathetic_dialogues,labels,Unnamed: 5,Unnamed: 6
0,0,I remember going to the fireworks with my best...,sentimental,Customer :I remember going to see the firework...,"Was this a friend you were in love with, or ju...",NaN,NaN
1,1,I remember going to the fireworks with my best...,sentimental,Customer :This was a best friend. I miss her.\...,Where has she gone?,NaN,NaN
2,2,I remember going to the fireworks with my best...,sentimental,Customer :We no longer talk.\nAgent :,Oh was this something that happened because of...,NaN,NaN
3,3,I remember going to the fireworks with my best...,sentimental,Customer :Was this a friend you were in love w...,This was a best friend. I miss her.,NaN,NaN
4,4,I remember going to the fireworks with my best...,sentimental,Customer :Where has she gone?\nAgent :,We no longer talk.,NaN,NaN


In [17]:
df.shape

(64636, 7)

In [18]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64636 entries, 0 to 64635
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Unnamed: 0            64636 non-null  int64 
 1   Situation             64636 non-null  object
 2   emotion               64632 non-null  object
 3   empathetic_dialogues  64636 non-null  object
 4   labels                64636 non-null  object
 5   Unnamed: 5            113 non-null    object
 6   Unnamed: 6            5 non-null      object
dtypes: int64(1), object(6)
memory usage: 3.5+ MB


In [19]:
df.isnull().sum()

,0
Unnamed: 0,0
Situation,0
emotion,4
empathetic_dialogues,0
labels,0
Unnamed: 5,64523
Unnamed: 6,64631


In [20]:
import pandas as pd

# 1. Google Drive direct download URL (for Colab)
file_id = '1AIjpubvoxiJOMz974J6omBMg82UDN0OW'
url = f'https://drive.google.com/uc?id={file_id}'

# 2. Load specifically the columns using the URL
cols = ['Situation', 'emotion', 'empathetic_dialogues', 'labels']
df = pd.read_csv(url, usecols=cols)

# 3. Rename Situation to text immediately for our tokenizer
df = df.rename(columns={'Situation': 'text'})

# 4. The "Catch-All" Mapping
mapping = {
    'sentimental': 'Mental Filter',
    'afraid': 'Catastrophizing',
    'proud': 'Neutral',
    'furious': 'Personalization',
    'guilty': 'Should Statements',
    'sad': 'Mental Filter',
    'lonely': 'Mental Filter',
    'terrified': 'Catastrophizing'
}

# 5. Clean strings and apply mapping
df['text'] = df['text'].astype(str).str.lower().str.strip()
df['emotion'] = df['emotion'].astype(str).str.lower().str.strip()
df['distortion'] = df['emotion'].map(mapping)

# 6. Drop rows where we couldn't find a distortion match
df_final = df.dropna(subset=['distortion']).copy()

print(f"✅ Success! We now have {len(df_final)} rows to train on.")
print("📊 Distribution of distortions:\n", df_final['distortion'].value_counts())


✅ Success! We now have 16605 rows to train on.
📊 Distribution of distortions:
 distortion
Mental Filter        6092
Catastrophizing      4168
Neutral              2247
Should Statements    2053
Personalization      2045
Name: count, dtype: int64


In [21]:
# Create the numeric labels for the model
df_final['label'] = df_final['distortion'].astype('category').cat.codes

# Save the mapping so we know what number belongs to what distortion
label_names = dict(enumerate(df_final['distortion'].astype('category').cat.categories))
print("🔢 Numeric Label Map:", label_names)

🔢 Numeric Label Map: {0: 'Catastrophizing', 1: 'Mental Filter', 2: 'Neutral', 3: 'Personalization', 4: 'Should Statements'}


In [22]:
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizer

In [23]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [24]:
tokenizer

DistilBertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [25]:
class DistortionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # MODERN WAY: Call the tokenizer directly
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
df_final['label'] = df_final['distortion'].astype('category').cat.codes

In [28]:
df_train, df_val = train_test_split(
    df_final, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_final['label']
)

# 3. Create the Dataset Objects
train_data = DistortionDataset(
    texts=df_train.text.to_numpy(),
    labels=df_train.label.to_numpy(),
    tokenizer=tokenizer
)

val_data = DistortionDataset(
    texts=df_val.text.to_numpy(),
    labels=df_val.label.to_numpy(),
    tokenizer=tokenizer
)

In [29]:
len(train_data)

13284

In [30]:
len(val_data)

3321

In [31]:
from transformers import DistilBertForSequenceClassification

num_classes = len(df_final['distortion'].unique())

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_classes
)

# Move model to GPU if you have one, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"🧠 Model loaded on {device} with {num_classes} output classes.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🧠 Model loaded on cuda with 5 output classes.


In [32]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # We use 'weighted' to account for class imbalance
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    
    return {
        'accuracy': acc,
        'f1': f1
    }

In [33]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',          
    num_train_epochs=3,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    warmup_steps=500,                
    weight_decay=0.01,               
    logging_dir='./logs',            
    logging_steps=100,
    eval_strategy="epoch",     # <--- CHANGED from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,     
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [34]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
)

print("🏎️ Training is starting... Grab a coffee, Mayank!")
trainer.train()

🏎️ Training is starting... Grab a coffee, Mayank!


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.231489,0.163062,0.958446,0.958635
2,0.057808,0.049827,0.987955,0.987973
3,0.011772,0.021021,0.994881,0.994881


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2493, training_loss=0.22168063663164392, metrics={'train_runtime': 578.729, 'train_samples_per_second': 68.861, 'train_steps_per_second': 4.308, 'total_flos': 1319843301626880.0, 'train_loss': 0.22168063663164392, 'epoch': 3.0})

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [37]:
import os

# Define a professional path on your Drive
save_path = "/content/drive/MyDrive/MentalHealth_AI_Model"

# Create the folder if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Save the model and tokenizer to your Drive
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model successfully anchored to Google Drive at: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model successfully anchored to Google Drive at: /content/drive/MyDrive/MentalHealth_AI_Model


In [38]:
!ls -lh "/content/drive/MyDrive/MentalHealth_AI_Model"

total 257M
-rw------- 1 root root  893 Mar 15 10:40 config.json
-rw------- 1 root root 256M Mar 15 10:40 model.safetensors
-rw------- 1 root root  328 Mar 15 10:40 tokenizer_config.json
-rw------- 1 root root 695K Mar 15 10:40 tokenizer.json


In [39]:
!pip install optimum[onnxruntime]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 101.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 107.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 24.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [40]:
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer
from pathlib import Path

save_path = "/content/drive/MyDrive/MentalHealth_AI_Model"
onnx_path = Path("/content/drive/MyDrive/MentalHealth_AI_Quantized")

# 1. Load the model as an ORT (ONNX Runtime) model
print("🚀 Converting to ONNX and Quantizing...")
model_ort = ORTModelForSequenceClassification.from_pretrained(save_path, export=True)

# 2. Save the ONNX model
model_ort.save_pretrained(onnx_path)

print(f"✅ Quantized model saved at: {onnx_path}")

ImportError: cannot import name 'FLAX_WEIGHTS_NAME' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

In [42]:
import shutil
from google.colab import files

# 1. Define the folder you want to download
# We'll download the Quantized version since that's the "Mobile-Ready" one
folder_to_zip = '/content/drive/MyDrive/MentalHealth_AI_Model'
output_filename = 'mental_health_model_mobile'

# 2. Zip the folder
print("📦 Zipping the model folder... please wait.")
shutil.make_archive(output_filename, 'zip', folder_to_zip)

# 3. Trigger the browser download to your laptop
print("🚀 Launching download to your laptop!")
files.download(f'{output_filename}.zip')

📦 Zipping the model folder... please wait.
🚀 Launching download to your laptop!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>